# Evaluation — RAGAS (baseline vs final) + 10-run metrics

**Topic 8 · AI governance.** Produces the numbers for `REPORT.md` §3.

- **RAGAS**: baseline retriever (plain TF-IDF) vs final pipeline (hybrid BM25+dense+RRF,
  parent-child, **real cross-encoder rerank on GPU**). Judge = **Mistral**; embeddings for
  `answer_relevancy` = **local HuggingFace** on the GPU (Mistral has no embeddings endpoint).
- **10 runs**: average cost (USD), average latency (s), tool-call distribution.

### Before you run
1. Runtime → **Change runtime type → GPU** (T4 is enough).
2. Your `MISTRAL_API_KEY` must be reachable: either in the repo `.env`, or add it via
   Colab **Secrets** (🔑 left panel) as `MISTRAL_API_KEY`.
3. Run cells top to bottom. If imports fail right after the install cell, **restart the
   runtime** (Runtime → Restart) and re-run from cell 2 (skip the install).

> Heads-up: this makes many Mistral calls (answer generation + RAGAS judge + 10 agent
> runs). With a rate-limited key it is **slow but auto-retried**; total cost ≈ \$0.3–0.6.

In [ ]:
# 1) Install the eval stack (pinned, same as Block 1 §6b) + local models for GPU
import sys, subprocess
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("ragas==0.1.21", "langchain==0.2.16", "langchain-community==0.2.16",
    "langchain-openai==0.1.25", "datasets", "sentence-transformers")
print("deps installed — if the imports below fail, restart the runtime and skip this cell.")

In [ ]:
# 2) Locate the repo, enable the REAL cross-encoder (GPU), load keys
import os, sys, json, time
from pathlib import Path

here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p/"src").exists() and (p/"data").exists()), None)
assert ROOT, "Open this notebook from inside the repo (must see src/ and data/)."
sys.path.insert(0, str(ROOT/"src")); os.chdir(ROOT)
print("repo:", ROOT)

# turn ON the real cross-encoder BEFORE importing retrieval (it reads this at import)
os.environ["USE_REAL_RERANKER"] = "1"
os.environ.setdefault("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")

# keys: .env first, then Colab secret fallback
try:
    from dotenv import load_dotenv; load_dotenv(ROOT/".env")
except Exception: pass
if not os.getenv("MISTRAL_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
    except Exception: pass
assert os.getenv("MISTRAL_API_KEY"), "Set MISTRAL_API_KEY in .env or Colab Secrets."
os.environ.setdefault("LLM_PROVIDER", "mistral")
os.environ.setdefault("LLM_MODEL", "mistral-large-latest")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — cross-encoder & embeddings will run on CPU (slower).")

In [ ]:
# 3) Build the two retrievers + load eval questions
import retrieval as R
import resilience; resilience.instrument_retries()   # retry on Mistral 429
from llm_helpers import make_client

CORPUS = R.CORPUS
ev = json.load(open(ROOT/"data"/"eval_questions.json", encoding="utf-8"))
QUESTIONS, GROUND_TRUTH = ev["questions"], ev["ground_truth"]
print(len(CORPUS), "passages,", len(QUESTIONS), "eval questions")

# BASELINE = plain TF-IDF (before any Block-1 improvement)
_baseline = R.TinyTfidf(CORPUS)
def baseline_retrieve(q, k=3): return [t for _, t in _baseline.search(q, k)]

# FINAL = hybrid + parent-child + REAL cross-encoder rerank
def final_retrieve(q, k=3): return R.production_retrieve(q, k_final=k)

In [ ]:
# 4) Warm up the cross-encoder on the GPU (first call downloads + loads the model)
t = time.time(); _ = final_retrieve(QUESTIONS[0], 3)
print(f"final_retrieve OK — cross-encoder loaded in {time.time()-t:.1f}s")
print("USE_REAL_RERANKER =", os.getenv("USE_REAL_RERANKER"))

In [ ]:
# 5) Generate an answer per (retriever, question) with Mistral — grounded in the contexts
client = make_client(quiet=True)
def generate_answer(question, contexts):
    ctx = "\n---\n".join(contexts)
    msg = [
        {"role": "system", "content":
         "Answer ONLY from the CONTEXT. Cite the article id. If the answer is not in the context, say so."},
        {"role": "user", "content": f"CONTEXT:\n{ctx}\n\nQUESTION: {question}"},
    ]
    return client.complete(msg, max_tokens=400).content or ""

def build_rows(retrieve):
    rows = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for q, gt in zip(QUESTIONS, GROUND_TRUTH):
        ctx = retrieve(q, 3)
        rows["question"].append(q); rows["contexts"].append(ctx)
        rows["ground_truth"].append(gt); rows["answer"].append(generate_answer(q, ctx))
        time.sleep(1)   # gentle on the rate limit
    return rows

base_rows = build_rows(baseline_retrieve)
final_rows = build_rows(final_retrieve)
print("answers generated for baseline and final")

In [ ]:
# 6) Configure RAGAS: Mistral judge (OpenAI-compatible) + local HF embeddings on GPU
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_recall, context_precision, faithfulness, answer_relevancy
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

judge = ChatOpenAI(model="mistral-large-latest", api_key=os.environ["MISTRAL_API_KEY"],
                   base_url="https://api.mistral.ai/v1", temperature=0, max_retries=6, timeout=120)
emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"})

METRICS = [context_recall, context_precision, faithfulness, answer_relevancy]
RC = RunConfig(max_workers=1, max_retries=10, max_wait=90, timeout=180)  # serial -> avoid 429

def run_ragas(rows):
    ds = Dataset.from_dict(rows)
    return evaluate(ds, metrics=METRICS, llm=judge, embeddings=emb,
                    run_config=RC, raise_exceptions=False)

In [ ]:
# 7) Run RAGAS on both (this is the slow part — serial calls + retries)
print("=== RAGAS — BASELINE (plain TF-IDF) ===")
base = run_ragas(base_rows); print(base)
print("\n=== RAGAS — FINAL (hybrid + real cross-encoder) ===")
final = run_ragas(final_rows); print(final)

In [ ]:
# 8) Comparison table -> paste into REPORT.md §3
def g(res, k):
    try: return round(float(res[k]), 3)
    except Exception: return "n/a"

names = ["context_recall", "context_precision", "faithfulness", "answer_relevancy"]
print(f'{"Metric":<20}{"Baseline":>10}{"Final":>10}')
print("-"*40)
for m in names:
    print(f"{m:<20}{str(g(base, m)):>10}{str(g(final, m)):>10}")
print("\n(answer_relevancy uses the local HF embeddings; the other 3 use the Mistral judge.)")

## Part B — 10 runs (average cost, latency, tool-call distribution) → REPORT §3

Runs the full agent (L1 → gated retrieval → self-consistency k=3 → critic) on 10 questions
and aggregates the numbers. Slower (each run ≈ 7 Mistral calls). Lower `RUN_QS` slicing if
you are short on rate-limit budget.

In [ ]:
# 9) 10 agent runs
from agent import run as agent_run

RUN_QS = (QUESTIONS + [
    "What is the maximum administrative fine under the GDPR?",
    "What is the right to erasure under the GDPR?",
    "Which AI systems are considered high-risk under the AI Act?",
    "What obligations apply to general-purpose AI models?",
    "Does the AI Act require human oversight for high-risk systems?",
])[:10]

results = []
for i, q in enumerate(RUN_QS, 1):
    t = time.time(); out = agent_run(q, verbose=False); dt = time.time() - t
    results.append({"latency": dt, "cost": out.get("cost_usd", 0.0),
                    "tools": out.get("tool_calls", {}), "verdict": out["critic"]["verdict"]})
    print(f"{i:2}. {dt:5.1f}s  ${out.get('cost_usd', 0):.4f}  tools={out.get('tool_calls', {})}  {out['critic']['verdict']}")

In [ ]:
# 10) Aggregate -> paste into REPORT.md §3
import statistics as st, collections
lat = [r["latency"] for r in results]; cost = [r["cost"] for r in results]
dist = collections.Counter()
for r in results:
    for tname, c in r["tools"].items(): dist[tname] += c

print("=== 10-run summary (REPORT §3) ===")
print(f"average latency : {st.mean(lat):.1f} s")
print(f"average cost    : ${st.mean(cost):.4f}  (total ${sum(cost):.3f})")
print(f"tool-call distribution (across {len(results)} runs): {dict(dist)}")
print(f"critic verdicts : {collections.Counter(r['verdict'] for r in results)}")